In [69]:
from pyspark.sql.types import DoubleType, FloatType, LongType, StructType,StructField, StringType

In [70]:
from datetime import datetime, timedelta

In [71]:
schema = StructType([
  StructField("id", LongType(), True),
  StructField("nome", StringType(), True),
  StructField("cargo", StringType(), True),
  StructField("salario", FloatType(), True),
  StructField("idade", LongType(), True)
])

In [72]:
df = spark.createDataFrame([], schema)

In [73]:
df.writeTo("empresa.rh.colaborador").create()

In [74]:
schema = spark.table("empresa.rh.colaborador").schema

In [75]:
data = [
    (1, "Teresa", "diretor", 20000.0, 48),
    (2, "Pedro", "diretor", 20000.0, 56),
    (3, "Joao", "gerente", 15000.0, 45),
    (4, "Maria", "analista", 8000.0, 32),
    (5, "Cesar", "analista", 8000.0, 30)
  ]

In [76]:
df = spark.createDataFrame(data, schema)

In [77]:
df.writeTo("empresa.rh.colaborador").append()

In [78]:
spark.table("empresa.rh.colaborador").show()

+---+------+--------+-------+-----+
| id|  nome|   cargo|salario|idade|
+---+------+--------+-------+-----+
|  1|Teresa| diretor|20000.0|   48|
|  2| Pedro| diretor|20000.0|   56|
|  3|  Joao| gerente|15000.0|   45|
|  4| Maria|analista| 8000.0|   32|
|  5| Cesar|analista| 8000.0|   30|
+---+------+--------+-------+-----+



In [79]:
data = [
    (6, "Jose", "estagiario", 1500.0, 20),
    (7, "Estela", "especialista", 12000.0, 42),
    (8, "Osmar", "estagiario", 1500.0, 18),
    (9, "Antonio", "especialista", 12000.0, 44),
    (10, "Savia", "especialista", 12000.0, 38)
  ]

In [80]:
df = spark.createDataFrame(data, schema)

In [81]:
df.writeTo("empresa.rh.colaborador").append()

In [82]:
dt_time_end = datetime.now()

In [83]:
dt_time_start = dt_time_end - timedelta(minutes=1)

In [84]:
spark.sql("SELECT * FROM empresa.rh.colaborador.snapshots").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-03-06 01:36:...|7389047449352305062|               NULL|   append|s3://warehouse/em...|{spark.app.id -> ...|
|2026-03-06 01:36:...|3024809035631702764|7389047449352305062|   append|s3://warehouse/em...|{spark.app.id -> ...|
|2026-03-06 01:37:...|4473420611620888253|3024809035631702764|   append|s3://warehouse/em...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [85]:
spark.sql("SELECT * FROM empresa.rh.colaborador.files").where("content = 0").show()

+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|    readable_metrics|
+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|      0|s3://warehouse/em...|    PARQUET|      0|           1|              1434|{1 -> 46, 2 -> 46...|{1 -> 1, 2 -> 1, ...|{1 -> 0, 2 -> 0, ...|        {4 -> 0

In [86]:
spark.sql("SELECT * FROM empresa.rh.colaborador FOR SYSTEM_TIME AS OF '{}'".format(dt_time_start)).show()

+---+----+-----+-------+-----+
| id|nome|cargo|salario|idade|
+---+----+-----+-------+-----+
+---+----+-----+-------+-----+



In [87]:
spark.sql("SELECT * FROM empresa.rh.colaborador FOR SYSTEM_TIME AS OF '{}'".format(dt_time_end)).show()

+---+-------+------------+-------+-----+
| id|   nome|       cargo|salario|idade|
+---+-------+------------+-------+-----+
|  1| Teresa|     diretor|20000.0|   48|
|  2|  Pedro|     diretor|20000.0|   56|
|  3|   Joao|     gerente|15000.0|   45|
|  4|  Maria|    analista| 8000.0|   32|
|  5|  Cesar|    analista| 8000.0|   30|
|  6|   Jose|  estagiario| 1500.0|   20|
|  7| Estela|especialista|12000.0|   42|
|  8|  Osmar|  estagiario| 1500.0|   18|
|  9|Antonio|especialista|12000.0|   44|
| 10|  Savia|especialista|12000.0|   38|
+---+-------+------------+-------+-----+



In [88]:
spark.sql("""
SELECT * FROM empresa.rh.colaborador FOR SYSTEM_TIME AS OF '{}'
EXCEPT
SELECT * FROM empresa.rh.colaborador FOR SYSTEM_TIME AS OF '{}'
""".format(dt_time_end, dt_time_start)).show()

+---+-------+------------+-------+-----+
| id|   nome|       cargo|salario|idade|
+---+-------+------------+-------+-----+
|  2|  Pedro|     diretor|20000.0|   56|
|  6|   Jose|  estagiario| 1500.0|   20|
|  3|   Joao|     gerente|15000.0|   45|
|  1| Teresa|     diretor|20000.0|   48|
|  5|  Cesar|    analista| 8000.0|   30|
|  8|  Osmar|  estagiario| 1500.0|   18|
|  7| Estela|especialista|12000.0|   42|
|  4|  Maria|    analista| 8000.0|   32|
|  9|Antonio|especialista|12000.0|   44|
| 10|  Savia|especialista|12000.0|   38|
+---+-------+------------+-------+-----+



In [89]:
spark.sql("UPDATE empresa.rh.colaborador SET cargo = 'assistente' where  cargo = 'analista'")

DataFrame[]

In [90]:
dt_time_end = datetime.now()

In [91]:
dt_time_start = dt_time_end - timedelta(minutes=1)

In [92]:
spark.sql("""
SELECT * FROM empresa.rh.colaborador FOR SYSTEM_TIME AS OF '{}'
EXCEPT
SELECT * FROM empresa.rh.colaborador FOR SYSTEM_TIME AS OF '{}'
""".format(dt_time_end, dt_time_start)).show()

+---+-------+------------+-------+-----+
| id|   nome|       cargo|salario|idade|
+---+-------+------------+-------+-----+
|  6|   Jose|  estagiario| 1500.0|   20|
|  9|Antonio|especialista|12000.0|   44|
|  8|  Osmar|  estagiario| 1500.0|   18|
|  7| Estela|especialista|12000.0|   42|
| 10|  Savia|especialista|12000.0|   38|
|  4|  Maria|  assistente| 8000.0|   32|
|  5|  Cesar|  assistente| 8000.0|   30|
+---+-------+------------+-------+-----+



In [93]:
spark.sql("DROP TABLE empresa.rh.colaborador")

DataFrame[]